## 04b - Topic Labeling and Export

Applies the final labeling to the **frozen** BERTopic model trained in 04a. No model is retrained here.

**Input**
- `data/processed/trenord_reviews_topics.csv`: Italian reviews with the topic id assigned by 04a
- `data/processed/bertopic_model/`: frozen BERTopic model (30 topics, ids 0-29)
- `data/processed/trenord_reviews_clean.csv`: used only to check that `reply_text` is unchanged

**Output**
- `data/processed/trenord_reviews_labeled.csv`: all Italian reviews with label and category (EXCLUDE included)
- `data/processed/trenord_reviews_analysis.csv`: informative topics only (EXCLUDE removed)
- `data/processed/top_reviews_per_topic.csv`: up to 5 most upvoted reviews per topic, for the dashboard
- `data/processed/bertopic_model_final/`: model after `merge_topics()`, with updated representations

**Steps**
1. Guard: verify that the model is the frozen one (ids 0-29, expected sizes).
2. Label map keyed on the frozen topic ids.
3. `merge_topics()` to update the topic representations, with a consistency check against the label map.
4. Final distribution, excluded-reviews bias check, export and output checks.

Only counts and aggregates are printed; review texts are never shown in this notebook.

In [1]:
# --- IMPORTS ---
import re

import pandas as pd
from bertopic import BERTopic
from bertopic.representation import MaximalMarginalRelevance

C:\Users\pablo\Documents\projects\trenord-analysis\venv_trenord\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- LOAD DATA AND FROZEN MODEL ---
df = pd.read_csv("../data/processed/trenord_reviews_topics.csv")
df["date"] = pd.to_datetime(df["date"])

# Topic modeling was performed on Italian reviews only
italian_df = df[df["language"] == "it"].copy()

topic_model = BERTopic.load(
    "../data/processed/bertopic_model",
    embedding_model="paraphrase-multilingual-mpnet-base-v2"
)

print(f"Italian reviews loaded: {len(italian_df):,}")


# --- GUARD: THE MODEL MUST BE THE FROZEN ONE ---
# The label map below is keyed on the topic ids of the frozen model.
# If 04a is retrained, ids and sizes change and every label would silently point to the wrong topic.
expected_sizes = {0: 1119, 1: 654, 2: 686, 3: 531, 4: 561, 5: 501, 6: 300, 7: 217, 8: 187, 9: 200,
                  10: 155, 11: 164, 12: 134, 13: 173, 14: 117, 15: 160, 16: 134, 17: 91, 18: 82, 19: 96,
                  20: 84, 21: 56, 22: 86, 23: 74, 24: 62, 25: 43, 26: 48, 27: 66, 28: 96, 29: 74}

csv_sizes = italian_df["topic"].value_counts().to_dict()
model_sizes = {}
for topic_id, size in topic_model.topic_sizes_.items():
    model_sizes[int(topic_id)] = int(size)

guard_checks = {
    "topic ids in topics.csv are exactly 0-29": sorted(csv_sizes) == list(range(30)),
    "topic ids in the model are exactly 0-29": sorted(model_sizes) == list(range(30)),
    "topic sizes in topics.csv match expected": csv_sizes == expected_sizes,
    "topic sizes in the model match expected": model_sizes == expected_sizes,
    "model assignments match topics.csv row by row": list(topic_model.topics_) == italian_df["topic"].tolist(),
}
for check_name, passed in guard_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {check_name}")

if not all(guard_checks.values()):
    raise RuntimeError(
        "The BERTopic model or topics.csv no longer matches the frozen model this labeling was built on. "
        "04a was probably retrained: topic ids and sizes have changed, so the label map below must be "
        "rebuilt from a new manual review of the topics before running this notebook."
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12556.85it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10630.26it/s]

Italian reviews loaded: 6,951
PASS  topic ids in topics.csv are exactly 0-29
PASS  topic ids in the model are exactly 0-29
PASS  topic sizes in topics.csv match expected
PASS  topic sizes in the model match expected
PASS  model assignments match topics.csv row by row


## Decision log

### Method
The 30 topics of the frozen model were reviewed manually, using for each topic its size, rating profile,
average length, MMR keywords, the 5 most representative reviews and a random sample of further reviews.
The working notes (`labeling/topic_review.md`) contain review texts, so they are kept local and not published.
Each topic received a short English label (2-3 words) and one of the categories of
Maalej et al. (2016), adapted to this dataset:

- **Problem**: something in the app does not work, or no longer works
- **Feature Request**: functionality that users ask to be added
- **Praise**: what users appreciate
- **Other**: not about the app itself (e.g. the rail service) or comparisons between versions

The label map is keyed on the **frozen topic ids**, not on the ids produced by `merge_topics()`,
because BERTopic renumbers all topics by size after a merge. Topics that share a label are merged.

### Final labeling (13 topics)

| Frozen id(s) | Label | Category |
|---|---|---|
| 0 | Ticket Purchase Failures | Problem |
| 1 | Subscription Issues | Problem |
| 2, 4, 8, 28 | Real-time Train Info | Problem |
| 3 | Poor Usability | Problem |
| 5, 16 | App Stability & Crashes | Problem |
| 7, 23 | Login & Account | Problem |
| 11 | Station & Favorites Bugs | Problem |
| 14 | QR Code at Turnstiles | Problem |
| 27 | Station Coverage Gaps | Problem |
| 6, 9 | Ease of Use | Praise |
| 13, 19 | Purchase & Payment Praise | Praise |
| 20 | Version Evolution | Other |
| 22 | Trenord Service Complaints | Other |
| 10, 12, 15, 17, 18, 21, 24, 25, 26, 29 | EXCLUDE | — |

Sizes and average ratings are computed in the cells below.

### Exclusion rule
A topic is excluded when it consists mostly of very short reviews (about five words or fewer) that express
generic sentiment ("ottima", "pessima", "non funziona") without saying what works or what fails.
Such reviews carry the rating but no information about the app, so they would only add noise to topic-level
comparisons. Excluded reviews remain in `trenord_reviews_labeled.csv` with label `EXCLUDE`.

### Notable decisions
- **Topic 3 is kept as Poor Usability, although it is borderline.** Its reviews are short and many are generic
  complaints, which would point to exclusion. It is nevertheless the only topic that collects complaints about
  clarity, intuitiveness, complexity and slowness of the app; excluding it would remove the usability dimension
  from the analysis entirely, and would make the excluded set even more positive than the included one.
  Results for this topic should be read with this caveat.
- **Station Coverage Gaps is a Problem, not a Feature Request.** The reviews report stations that were removed
  from the app or are missing from its station list, often after an update: a regression in existing
  functionality rather than a request for something new.
- **No topic qualifies as Feature Request.** Requests do occur in the reviews, but they are scattered across
  topics and do not dominate any of them. Topic 19, previously labeled "Digital Payments" (Feature Request),
  consists mostly of positive reviews of the Satispay and PayPal payment options and is merged with topic 13
  into Purchase & Payment Praise.
- **Delay alerts are part of Real-time Train Info.** Topics 8 and 28 (late or wrong delay information) were
  previously a separate "Delay Notifications" topic; their reviews describe the same problem as topics 2 and 4.

### Bias note
Excluded topics are mostly short positive reviews, so excluding them can make the analysis set more negative
than the full dataset. The size and direction of this effect are computed in the bias check below.

In [3]:
# --- LABEL MAP (keyed on frozen topic ids) ---
# Each entry: (frozen topic ids, label, category). Topics sharing a label are merged in the next cell.
EXCLUDE_LABEL = "EXCLUDE"

LABEL_GROUPS = [
    ([0],            "Ticket Purchase Failures",   "Problem"),
    ([1],            "Subscription Issues",        "Problem"),
    ([2, 4, 8, 28],  "Real-time Train Info",       "Problem"),
    ([3],            "Poor Usability",             "Problem"),
    ([5, 16],        "App Stability & Crashes",    "Problem"),
    ([6, 9],         "Ease of Use",                "Praise"),
    ([7, 23],        "Login & Account",            "Problem"),
    ([11],           "Station & Favorites Bugs",   "Problem"),
    ([13, 19],       "Purchase & Payment Praise",  "Praise"),
    ([14],           "QR Code at Turnstiles",      "Problem"),
    ([20],           "Version Evolution",          "Other"),
    ([22],           "Trenord Service Complaints", "Other"),
    ([27],           "Station Coverage Gaps",      "Problem"),
    ([10, 12, 15, 17, 18, 21, 24, 25, 26, 29], EXCLUDE_LABEL, None),
]

# Every frozen id must appear exactly once in the map
label_by_id = {}
category_by_id = {}
id_occurrences = []
for topic_ids, label, category in LABEL_GROUPS:
    for topic_id in topic_ids:
        id_occurrences.append(topic_id)
        label_by_id[topic_id] = label
        category_by_id[topic_id] = category

assert sorted(id_occurrences) == list(range(30)), "Every topic id 0-29 must appear exactly once in LABEL_GROUPS."

italian_df["topic_label"] = italian_df["topic"].map(label_by_id)
italian_df["category"] = italian_df["topic"].map(category_by_id)

n_labels = italian_df.loc[italian_df["topic_label"] != EXCLUDE_LABEL, "topic_label"].nunique()
print(f"Label map covers {len(label_by_id)} topic ids, each exactly once")
print(f"Final labels (excluding EXCLUDE): {n_labels}")

Label map covers 30 topic ids, each exactly once
Final labels (excluding EXCLUDE): 13


In [4]:
# --- MERGE TOPICS (representations only) ---
# Labels are already assigned from the frozen ids above. merge_topics() is run only to obtain updated
# keywords and topic embeddings for the saved final model.
TOPICS_TO_MERGE = [[2, 4, 8, 28], [5, 16], [7, 23], [6, 9], [13, 19]]

# The merge list must be exactly the multi-topic groups of the label map
groups_from_map = []
for topic_ids, label, category in LABEL_GROUPS:
    if label != EXCLUDE_LABEL and len(topic_ids) > 1:
        groups_from_map.append(sorted(topic_ids))
merge_groups = []
for group in TOPICS_TO_MERGE:
    merge_groups.append(sorted(group))
assert sorted(merge_groups) == sorted(groups_from_map), "TOPICS_TO_MERGE does not match LABEL_GROUPS."

# merge_topics() needs the documents in the same order as the model assignments.
# BERTopic cannot handle empty strings, so they are replaced with a single space.
texts = italian_df["text_clean"].tolist()
for i in range(len(texts)):
    text = texts[i]
    is_valid = isinstance(text, str) and len(text.strip()) > 0
    if not is_valid:
        texts[i] = " "

# The saved model does not store its representation model; MMR is restored with the same
# diversity used in 04a so that the merged keywords are computed consistently.
topic_model.representation_model = MaximalMarginalRelevance(diversity=0.3)
topic_model.merge_topics(texts, TOPICS_TO_MERGE)
italian_df["topic_merged"] = topic_model.topics_

# Each post-merge topic must contain reviews of a single label
labels_per_merged = italian_df.groupby("topic_merged")["topic_label"].nunique()
assert (labels_per_merged == 1).all(), "A post-merge topic contains reviews with different labels."

# Each kept label must correspond to exactly one post-merge topic
kept = italian_df[italian_df["topic_label"] != EXCLUDE_LABEL]
merged_per_label = kept.groupby("topic_label")["topic_merged"].nunique()
assert (merged_per_label == 1).all(), "A label is split across several post-merge topics."

merged_listing = (
    italian_df
    .groupby("topic_merged")
    .agg(size=("topic_label", "size"), label=("topic_label", "first"))
    .reset_index()
)
print(f"Topics after merge: {len(merged_listing)}")
print("PASS  every post-merge topic maps to exactly one label")
print("PASS  every kept label maps to exactly one post-merge topic")
print()
for _, row in merged_listing.iterrows():
    print(f"  ID {row['topic_merged']:3d} | {row['size']:4d} reviews | {row['label']}")

Topics after merge: 23
PASS  every post-merge topic maps to exactly one label
PASS  every kept label maps to exactly one post-merge topic

  ID   0 | 1530 reviews | Real-time Train Info
  ID   1 | 1119 reviews | Ticket Purchase Failures
  ID   2 |  654 reviews | Subscription Issues
  ID   3 |  635 reviews | App Stability & Crashes
  ID   4 |  531 reviews | Poor Usability
  ID   5 |  500 reviews | Ease of Use
  ID   6 |  291 reviews | Login & Account
  ID   7 |  269 reviews | Purchase & Payment Praise
  ID   8 |  164 reviews | Station & Favorites Bugs
  ID   9 |  160 reviews | EXCLUDE
  ID  10 |  155 reviews | EXCLUDE
  ID  11 |  134 reviews | EXCLUDE
  ID  12 |  117 reviews | QR Code at Turnstiles
  ID  13 |   91 reviews | EXCLUDE
  ID  14 |   86 reviews | Trenord Service Complaints
  ID  15 |   84 reviews | Version Evolution
  ID  16 |   82 reviews | EXCLUDE
  ID  17 |   74 reviews | EXCLUDE
  ID  18 |   66 reviews | Station Coverage Gaps
  ID  19 |   62 reviews | EXCLUDE
  ID  20 |  

In [5]:
# --- FINAL TOPIC DISTRIBUTION ---
df_analysis = italian_df[italian_df["topic_label"] != EXCLUDE_LABEL].copy()

n_reviews = len(df_analysis)
n_excluded = len(italian_df) - n_reviews

print(f"Topics for analysis : {df_analysis['topic_label'].nunique()}")
print(f"Reviews for analysis: {n_reviews:,}")
print(f"Reviews excluded    : {n_excluded:,} ({n_excluded / len(italian_df) * 100:.1f}% of Italian reviews)")
print()

distribution = (
    df_analysis
    .groupby(["topic_label", "category"])
    .agg(count=("rating", "size"), avg_rating=("rating", "mean"))
    .reset_index()
    .sort_values("count", ascending=False)
)
print(f"{'Topic':<28} {'Reviews':>7} {'Share':>7} {'Avg':>6}  Category")
print("-" * 66)
for _, row in distribution.iterrows():
    share = row["count"] / n_reviews * 100
    print(f"{row['topic_label']:<28} {row['count']:7d} {share:6.1f}% {row['avg_rating']:5.2f}★  {row['category']}")

print()
category_counts = df_analysis["category"].value_counts()
for category, count in category_counts.items():
    print(f"{category:<10} {count:5d} reviews ({count / n_reviews * 100:.1f}%)")

Topics for analysis : 13
Reviews for analysis: 6,046
Reviews excluded    : 905 (13.0% of Italian reviews)

Topic                        Reviews   Share    Avg  Category
------------------------------------------------------------------
Real-time Train Info            1530   25.3%  1.78★  Problem
Ticket Purchase Failures        1119   18.5%  1.41★  Problem
Subscription Issues              654   10.8%  1.44★  Problem
App Stability & Crashes          635   10.5%  1.58★  Problem
Poor Usability                   531    8.8%  1.45★  Problem
Ease of Use                      500    8.3%  4.53★  Praise
Login & Account                  291    4.8%  1.39★  Problem
Purchase & Payment Praise        269    4.4%  4.01★  Praise
Station & Favorites Bugs         164    2.7%  1.74★  Problem
QR Code at Turnstiles            117    1.9%  1.60★  Problem
Trenord Service Complaints        86    1.4%  1.14★  Other
Version Evolution                 84    1.4%  3.65★  Other
Station Coverage Gaps             66  

In [6]:
# --- VERIFY EXCLUDED REVIEWS BIAS ---
# If excluded reviews have a different rating profile, the analysis set is not representative
# of all Italian reviews, and this must be stated when interpreting results.
excluded = italian_df[italian_df["topic_label"] == EXCLUDE_LABEL]

excluded_avg = excluded["rating"].mean()
included_avg = df_analysis["rating"].mean()
all_avg = italian_df["rating"].mean()
difference = excluded_avg - included_avg

print("Excluded reviews by year:")
print(excluded["year"].value_counts().sort_index().to_string())
print()
print(f"Avg rating — excluded       : {excluded_avg:.2f}")
print(f"Avg rating — included       : {included_avg:.2f}")
print(f"Avg rating — all Italian    : {all_avg:.2f}")
print(f"Share of 4-5★ — excluded    : {(excluded['rating'] >= 4).mean() * 100:.1f}%")
print(f"Share of 4-5★ — included    : {(df_analysis['rating'] >= 4).mean() * 100:.1f}%")
print()

# A difference below 0.1 stars is treated as negligible
if abs(difference) < 0.1:
    print(f"Conclusion: excluded and included reviews have a similar average rating "
          f"({excluded_avg:.2f} vs {included_avg:.2f}); the exclusion does not bias the rating profile.")
elif difference > 0:
    print(f"Conclusion: excluded reviews are more positive than included ones "
          f"({excluded_avg:.2f} vs {included_avg:.2f}, +{difference:.2f}★).")
    print(f"The analysis set ({included_avg:.2f}★) is therefore more negative than all Italian reviews "
          f"({all_avg:.2f}★); this must be considered when interpreting results.")
else:
    print(f"Conclusion: excluded reviews are more negative than included ones "
          f"({excluded_avg:.2f} vs {included_avg:.2f}, {difference:.2f}★).")
    print(f"The analysis set ({included_avg:.2f}★) is therefore more positive than all Italian reviews "
          f"({all_avg:.2f}★); this must be considered when interpreting results.")

Excluded reviews by year:
year
2015     23
2016     28
2017     17
2018     31
2019    178
2020    165
2021    109
2022    118
2023     81
2024     59
2025     71
2026     25

Avg rating — excluded       : 3.67
Avg rating — included       : 1.94
Avg rating — all Italian    : 2.17
Share of 4-5★ — excluded    : 68.2%
Share of 4-5★ — included    : 19.1%

Conclusion: excluded reviews are more positive than included ones (3.67 vs 1.94, +1.73★).
The analysis set (1.94★) is therefore more negative than all Italian reviews (2.17★); this must be considered when interpreting results.


In [7]:
# --- SAVE ALL OUTPUTS ---
# 1. trenord_reviews_labeled.csv  : all Italian reviews with labels and categories
# 2. trenord_reviews_analysis.csv : informative topics only, EXCLUDE removed
# 3. top_reviews_per_topic.csv    : up to 5 most upvoted reviews per topic (dashboard Deep Dive tab)
# 4. bertopic_model_final         : BERTopic model after the merges
italian_df.to_csv("../data/processed/trenord_reviews_labeled.csv", index=False, encoding="utf-8-sig")
df_analysis.to_csv("../data/processed/trenord_reviews_analysis.csv", index=False, encoding="utf-8-sig")

# Only reviews with at least one upvote; the 5 most upvoted per topic
top_reviews = (
    df_analysis[df_analysis["thumbs_up"] > 0]
    .sort_values("thumbs_up", ascending=False)
    .groupby("topic_label")
    .head(5)
    [["topic_label", "category", "rating", "thumbs_up", "date", "text"]]
    .sort_values(["topic_label", "thumbs_up"], ascending=[True, False])
)
top_reviews.to_csv("../data/processed/top_reviews_per_topic.csv", index=False, encoding="utf-8-sig")

topic_model.save(
    "../data/processed/bertopic_model_final",
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model="paraphrase-multilingual-mpnet-base-v2"
)

print(f"trenord_reviews_labeled.csv  : {len(italian_df):,} reviews")
print(f"trenord_reviews_analysis.csv : {len(df_analysis):,} reviews")
print(f"top_reviews_per_topic.csv    : {len(top_reviews)} reviews across {top_reviews['topic_label'].nunique()} topics")
print("bertopic_model_final         : saved")

trenord_reviews_labeled.csv  : 6,951 reviews
trenord_reviews_analysis.csv : 6,046 reviews
top_reviews_per_topic.csv    : 65 reviews across 13 topics
bertopic_model_final         : saved


In [8]:
# --- OUTPUT CHECKS (counts only) ---
# The saved files are read back and checked for privacy and consistency.
# reply_text is compared with trenord_reviews_clean.csv, whose reply_text 01b verifies against the raw file;
# the raw file itself is not read downstream of 01b.
labeled_out = pd.read_csv("../data/processed/trenord_reviews_labeled.csv")
analysis_out = pd.read_csv("../data/processed/trenord_reviews_analysis.csv")
top_out = pd.read_csv("../data/processed/top_reviews_per_topic.csv")
clean_df = pd.read_csv("../data/processed/trenord_reviews_clean.csv")

HASHED_ID_PATTERN = re.compile(r"^[0-9a-f]{12}$")

ids_hashed = 0
for review_id in labeled_out["review_id"].astype(str):
    if HASHED_ID_PATTERN.match(review_id):
        ids_hashed += 1

replies = labeled_out[["review_id", "reply_text"]].merge(
    clean_df[["review_id", "reply_text"]], on="review_id", how="left", suffixes=("", "_clean")
)
replies_different = 0
for reply, reply_clean in zip(replies["reply_text"], replies["reply_text_clean"]):
    both_missing = pd.isna(reply) and pd.isna(reply_clean)
    if not both_missing and reply != reply_clean:
        replies_different += 1

output_checks = {
    "no author column in any output": "author" not in labeled_out.columns
                                      and "author" not in analysis_out.columns
                                      and "author" not in top_out.columns,
    "review_id hashed (12 hex) for every row": ids_hashed == len(labeled_out),
    "reply_text identical to clean.csv for every row": replies_different == 0,
    "analysis.csv has 13 labels": analysis_out["topic_label"].nunique() == 13,
    "analysis.csv has no EXCLUDE rows": (analysis_out["topic_label"] != EXCLUDE_LABEL).all(),
}

print(f"Rows labeled / analysis / top : {len(labeled_out):,} / {len(analysis_out):,} / {len(top_out)}")
print(f"review_id matching 12-hex     : {ids_hashed:,} / {len(labeled_out):,}")
print(f"reply_text different          : {replies_different}")
print()
print("Rows per label (analysis.csv):")
print(analysis_out["topic_label"].value_counts().to_string())
print()
print("Rows per category (analysis.csv):")
print(analysis_out["category"].value_counts().to_string())
print()
for check_name, passed in output_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {check_name}")

if not all(output_checks.values()):
    raise AssertionError("Output checks failed.")

Rows labeled / analysis / top : 6,951 / 6,046 / 65
review_id matching 12-hex     : 6,951 / 6,951
reply_text different          : 0

Rows per label (analysis.csv):
topic_label
Real-time Train Info          1530
Ticket Purchase Failures      1119
Subscription Issues            654
App Stability & Crashes        635
Poor Usability                 531
Ease of Use                    500
Login & Account                291
Purchase & Payment Praise      269
Station & Favorites Bugs       164
QR Code at Turnstiles          117
Trenord Service Complaints      86
Version Evolution               84
Station Coverage Gaps           66

Rows per category (analysis.csv):
category
Problem    5107
Praise      769
Other       170

PASS  no author column in any output
PASS  review_id hashed (12 hex) for every row
PASS  reply_text identical to clean.csv for every row
PASS  analysis.csv has 13 labels
PASS  analysis.csv has no EXCLUDE rows
